<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/CycleGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CycleGAN : transformer une image d'un domaine vers un autre

Dans ce notebook, on ne génère pas une image à partir d'un seed, et on ne modifie pas une image avec un texte.

Ici, on apprend une autre idée :

> transformer une image d'un **domaine visuel** vers un autre.

## Exemple
On peut par exemple transformer :
- une image d'été en image d'hiver ;
- une image d'hiver en image d'été.

## Objectif du notebook
Nous allons :
1. comprendre ce qu'est un domaine ;
2. télécharger un modèle préentraîné ;
3. appliquer ce modèle à des images ;
4. comparer les résultats avant / après.

## Ce qu'on veut comprendre
- ce qu'est une traduction d'image à image ;
- ce que signifie "sans paires alignées" ;
- pourquoi CycleGAN est différent de StyleGAN et StyleCLIP.

## Où se situe CycleGAN dans le cours ?

Jusqu'ici :

- **StyleGAN2** : générer une image ;
- **StyleGAN3** : faire évoluer une image de façon continue ;
- **StyleCLIP** : modifier une image selon un texte.

Maintenant :

- **CycleGAN** : transformer une image d'un domaine A vers un domaine B.

## Idée clé
Le point de départ n'est plus un seed ni un texte.

Le point de départ est une **image existante**.

## Qu'est-ce qu'un domaine visuel ?

Un domaine visuel est un ensemble d'images qui partagent un même type d'apparence.

Par exemple :

- domaine A : paysages d'été ;
- domaine B : paysages d'hiver.

## Ce que veut faire CycleGAN
Le modèle apprend à transformer :
- une image du domaine A en image du domaine B ;
- et aussi une image du domaine B en image du domaine A.

## Pourquoi est-ce intéressant ?

CycleGAN est utile quand on n'a pas besoin d'images parfaitement appariées.

Par exemple, on n'a pas besoin d'avoir :
- exactement la même scène en été ;
- exactement la même scène en hiver.

Il suffit d'avoir :
- un ensemble d'images d'été ;
- un ensemble d'images d'hiver.

C'est ce qu'on appelle une traduction **non appariée**.

In [ ]:
!pip -q install dominate

In [ ]:
!nvidia-smi || true

import sys
import platform

print("Python :", sys.version)
print("Plateforme :", platform.platform())

## Installer l'environnement

On récupère ici le dépôt officiel PyTorch de CycleGAN et Pix2Pix,
puis on installe les dépendances utiles.

In [ ]:
!rm -rf /content/pytorch-CycleGAN-and-pix2pix
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix /content/pytorch-CycleGAN-and-pix2pix
%cd /content/pytorch-CycleGAN-and-pix2pix
!pip -q install -r requirements.txt

## Télécharger un jeu de données d'exemple

Nous allons utiliser ici le jeu de données :

**summer2winter_yosemite**

Il contient :
- des images du domaine A : été ;
- des images du domaine B : hiver.

## Pourquoi ce choix ?
Parce que la transformation est visuellement très claire et facile à comprendre.

In [ ]:
!bash ./datasets/download_cyclegan_dataset.sh summer2winter_yosemite

## Télécharger un modèle préentraîné

Nous n'allons pas entraîner CycleGAN nous-mêmes.

Nous allons utiliser un modèle déjà entraîné sur ce jeu de données.

## Important
Comme pour StyleGAN, cela signifie :
- on ne montre pas ici l'entraînement complet ;
- on montre surtout comment utiliser un modèle appris.

In [ ]:
!bash ./scripts/download_cyclegan_model.sh summer2winter_yosemite

## Que va faire la commande suivante ?

Nous allons demander au modèle de prendre les images du dossier :

`testA`

c'est-à-dire les images d'été,

et de les transformer dans le style du domaine B,
c'est-à-dire l'hiver.

In [ ]:
!python test.py \
  --dataroot ./datasets/summer2winter_yosemite \
  --name summer2winter_yosemite_pretrained \
  --model test \
  --no_dropout

## Où sont enregistrés les résultats ?

Les images générées sont stockées dans un dossier `results`.

Nous allons maintenant aller regarder :
- les images d'entrée ;
- les images transformées.

In [ ]:
import glob

real_images = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/summer2winter_yosemite_pretrained/test_latest/images/*_real.png"))
fake_images = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/summer2winter_yosemite_pretrained/test_latest/images/*_fake.png"))

print("Nombre d'images réelles :", len(real_images))
print("Nombre d'images générées :", len(fake_images))

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

real_images = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/summer2winter_yosemite_pretrained/test_latest/images/*_real.png"))
fake_images = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/summer2winter_yosemite_pretrained/test_latest/images/*_fake.png"))

print("Nombre d'images réelles :", len(real_images))
print("Nombre d'images générées :", len(fake_images))

n_examples = 3

plt.figure(figsize=(12, 4 * n_examples))

for i in range(n_examples):
    img_in = Image.open(real_images[i]).convert("RGB")
    img_out = Image.open(fake_images[i]).convert("RGB")

    plt.subplot(n_examples, 2, 2*i + 1)
    plt.imshow(img_in)
    plt.title(f"Entrée {i+1} : été")
    plt.axis("off")

    plt.subplot(n_examples, 2, 2*i + 2)
    plt.imshow(img_out)
    plt.title(f"Sortie {i+1} : hiver")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Que transforme exactement CycleGAN ?

Le modèle ne remplace pas totalement l'image.

En général, il essaie de :
- conserver la structure globale ;
- modifier les textures, les couleurs, l'ambiance visuelle.

## Exemple ici
- les montagnes restent des montagnes ;
- les arbres restent des arbres ;
- mais la lumière, les couleurs et l'atmosphère changent.

## Pourquoi ce n'est pas la même chose que StyleGAN ?

Avec **StyleGAN** :
- on génère une image à partir d'un latent.

Avec **CycleGAN** :
- on transforme une image déjà existante.

Donc ici, il ne s'agit pas de "faire apparaître" une image,
mais de **traduire** une image dans un autre univers visuel.

## Deuxième exemple : horse -> zebra

Nous allons maintenant tester un autre exemple, souvent plus spectaculaire :

- domaine A : chevaux
- domaine B : zèbres

## Pourquoi cet exemple est-il intéressant ?
Il montre très bien que CycleGAN peut transformer l'apparence d'un objet
tout en conservant sa forme générale.

## Ce qu'il faut observer
- la silhouette reste-t-elle celle d'un cheval ?
- qu'est-ce qui change exactement ?
- le modèle transforme-t-il vraiment l'animal, ou surtout sa texture visuelle ?

In [ ]:
!bash ./datasets/download_cyclegan_dataset.sh horse2zebra
!bash ./scripts/download_cyclegan_model.sh horse2zebra

## Appliquer le modèle horse2zebra

On prend ici les images du dossier `testA`, c'est-à-dire les images du domaine A
(les chevaux), et on les transforme vers le domaine B (les zèbres).

In [ ]:
!python test.py \
  --dataroot ./datasets/horse2zebra \
  --name horse2zebra_pretrained \
  --model test \
  --no_dropout

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

horse_inputs = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/datasets/horse2zebra/testA/*.jpg"))
zebra_outputs = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/horse2zebra_pretrained/test_latest/images/*_fake.png"))

print("Nombre d'images d'entrée :", len(horse_inputs))
print("Nombre d'images générées :", len(zebra_outputs))

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

real_images = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/horse2zebra_pretrained/test_latest/images/*_real.png"))
fake_images = sorted(glob.glob("/content/pytorch-CycleGAN-and-pix2pix/results/horse2zebra_pretrained/test_latest/images/*_fake.png"))

print("Images réelles :", len(real_images))
print("Images générées :", len(fake_images))

n_examples = 3

plt.figure(figsize=(12, 4 * n_examples))

for i in range(n_examples):
    img_in = Image.open(real_images[i]).convert("RGB")
    img_out = Image.open(fake_images[i]).convert("RGB")

    plt.subplot(n_examples, 2, 2*i + 1)
    plt.imshow(img_in)
    plt.title(f"Entrée {i+1} : cheval")
    plt.axis("off")

    plt.subplot(n_examples, 2, 2*i + 2)
    plt.imshow(img_out)
    plt.title(f"Sortie {i+1} : zèbre")
    plt.axis("off")

plt.tight_layout()
plt.show()